# Tutorial: Build And Inspect A Durable Decisiveness Benchmark Release

This notebook walks through one frozen decisive-belief benchmark release end to end.

By the end you should have the four core objects that most downstream workflows need:
- `train_df`
- `train_target`
- `test_df`
- `test_target`

The release is defined by a durable decisiveness rule on the target market's own probability path.
A market becomes decisive on the YES side once `yes_probability >= tau_decisive` and remains there through resolution, or on the NO side once `yes_probability <= 1 - tau_decisive` and remains there through resolution.

The benchmark artifact itself is intentionally simple:
- `decisiveness_benchmark.examples`: the frozen manifest of pre-decisive prefixes
- `decisiveness_benchmark.market_timeseries`: normalized market-level probability histories


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from polymarket_research.benchmarks import DecisivenessBenchmark, DecisivenessBenchmarkConfig
from polymarket_research.data.canonical import CanonicalDataset
from polymarket_research.utils.filesystem import setup_root

REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'
DECISIVE_THRESHOLD = 0.95
RELEASE_NAME = f'{DATA_SOURCE}-decisive-belief-tau{int(DECISIVE_THRESHOLD * 100)}'
BENCHMARK_CACHE_DIR = ARTEFACT_ROOT / f'decisiveness_benchmark_tau{int(DECISIVE_THRESHOLD * 100)}'
SHOW_PROGRESS = True
USE_BENCHMARK_CACHE = True
BENCHMARK_CONFIG = DecisivenessBenchmarkConfig(
    decisive_threshold=DECISIVE_THRESHOLD,
    sample_every_hours=12,
    min_history_points=24,
    min_prefix_age_hours=6.0,
    min_time_to_decisive_hours=1.0,
    ordinal_bin_edges_hours=(24.0, 72.0),
    ordinal_bin_labels=('short', 'medium', 'long'),
    split_on='decisive_timestamp_utc',
    train_fraction=0.8,
    show_progress=SHOW_PROGRESS,
)

pd.set_option('display.max_colwidth', None)


## Step 1: Build Or Load One Release

Load the cached canonical dataset, then either reuse a cached decisive-belief release or build it from scratch.

The important idea is that one notebook run corresponds to one release rule: a chosen durable decisiveness threshold plus one ordinal horizon scheme.


In [ ]:
canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
print("Loaded canonical from parquet cache:", CANONICAL_CACHE_DIR)


In [ ]:
if USE_BENCHMARK_CACHE and (BENCHMARK_CACHE_DIR / "examples.parquet").exists():
    decisiveness_benchmark = DecisivenessBenchmark.load(BENCHMARK_CACHE_DIR, canonical=canonical)
    print(f"Loaded {RELEASE_NAME} from parquet cache:", BENCHMARK_CACHE_DIR)
else:
    decisiveness_benchmark = DecisivenessBenchmark.build(canonical, config=BENCHMARK_CONFIG)
    decisiveness_benchmark.save(BENCHMARK_CACHE_DIR)
    print(f"Built {RELEASE_NAME} from canonical")


In [ ]:
display(canonical.summary())
manifest = decisiveness_benchmark.manifest()
manifest_frame = pd.DataFrame([
    {"field": key, "value": repr(value) if isinstance(value, (list, dict, tuple)) else value}
    for key, value in manifest.items()
])
display(manifest_frame)

display(pd.DataFrame([
    {
        "release_name": decisiveness_benchmark.release_name,
        "examples": len(decisiveness_benchmark.examples),
        "eligible_markets": decisiveness_benchmark.examples["market_id"].nunique(),
        "market_timeseries_rows": len(decisiveness_benchmark.market_timeseries),
        "market_timeseries_columns": list(decisiveness_benchmark.market_timeseries.columns),
    }
]))


## Step 2: Materialize The Four Main Objects

For most workflows, this is the shortest useful path through the API:
- `train_df`: frozen train manifest rows
- `train_target`: train labels and auxiliary hours-to-decisive targets
- `test_df`: frozen test manifest rows
- `test_target`: held-out labels and auxiliary targets

`market_timeseries` stays separate and normalized by `market_id`. Treat it as source history for inspection, not as a materialized per-example feature panel.


In [ ]:
train_df = decisiveness_benchmark.split_examples("train")
train_target = decisiveness_benchmark.targets("train")
test_df = decisiveness_benchmark.split_examples("test")
test_target = decisiveness_benchmark.targets("test")

display(pd.DataFrame([{
    "train_examples": len(train_df),
    "test_examples": len(test_df),
    "train_targets": len(train_target),
    "test_targets": len(test_target),
}]))


In [ ]:
display(train_df.head())
display(train_target.head())
display(test_df.head())
display(test_target.head())

preview_market_ids = train_df["market_id"].drop_duplicates().head(3)
train_timeseries_preview = decisiveness_benchmark.market_timeseries.loc[
    lambda df: df["market_id"].isin(preview_market_ids)
].reset_index(drop=True)
display(train_timeseries_preview.head(20))


## Step 3: Inspect One Market End-To-End

The benchmark keeps the manifest rows and market history separate on purpose.

The workflow is:
1. pick one held-out `(market_id, cutoff_timestamp_utc)` snapshot
2. resolve its frozen manifest row
3. fetch the target market's full history
4. clip that history at the benchmark cutoff
5. compare the cutoff to the durable decisive entry time


In [ ]:
sample_row = test_df.sample(1, random_state=7).iloc[0]
market_id = sample_row["market_id"]
cutoff_timestamp_utc = pd.Timestamp(sample_row["cutoff_timestamp_utc"])
example = decisiveness_benchmark.resolve_market_snapshot(market_id, cutoff_timestamp_utc)
observation = decisiveness_benchmark.history_until(market_id, cutoff_timestamp_utc)
full_history = decisiveness_benchmark.market_history(market_id)

display(pd.DataFrame([{
    "market_id": example["market_id"],
    "cutoff_timestamp_utc": example["cutoff_timestamp_utc"],
    "decisive_timestamp_utc": example["decisive_timestamp_utc"],
    "decisive_side": example["decisive_side"],
    "hours_to_decisive": example["hours_to_decisive"],
    "label_name": example["label_name"],
    "prefix_rows": example["prefix_rows"],
}]))
display(example.to_frame(name="value"))
display(observation.tail(20))
display(
    test_target.loc[
        lambda df: (df["market_id"] == market_id) & (df["cutoff_timestamp_utc"] == cutoff_timestamp_utc)
    ]
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
full_history.set_index("timestamp_utc")["yes_probability"].plot(
    ax=ax, color="#ADB5BD", linewidth=1.5, label="full history"
)
observation.set_index("timestamp_utc")["yes_probability"].plot(
    ax=ax, color="#C92A2A", linewidth=2.5, label="visible prefix"
)
ax.axvline(example["cutoff_timestamp_utc"], color="#1C7ED6", linestyle="--", linewidth=1.5, label="cutoff")
ax.axvline(example["decisive_timestamp_utc"], color="#2B8A3E", linestyle="--", linewidth=1.5, label="durable decisive time")
ax.axhline(DECISIVE_THRESHOLD, color="#2B8A3E", linestyle=":", linewidth=1.2)
ax.axhline(1.0 - DECISIVE_THRESHOLD, color="#F08C00", linestyle=":", linewidth=1.2)
ax.set_ylim(0.0, 1.0)
ax.set_title(f"Prefix and durable decisive entry for {market_id}")
ax.set_xlabel("timestamp_utc")
ax.set_ylabel("yes_probability")
ax.legend()
plt.tight_layout()
plt.show()


## Step 4: Build A Lightweight Reference View

The frozen release artifact is the primary object. When you want feature-level diagnostics about convergence, derive the reference view explicitly.


In [ ]:
decisiveness_view = decisiveness_benchmark.build_reference_view()

display(decisiveness_view.head())

decisive_by_slice = (
    decisiveness_view.groupby(['research_category', 'confidence_slice', 'label_name'], dropna=False)
    .agg(
        rows=('example_id', 'size'),
        mean_hours_to_decisive=('hours_to_decisive', 'mean'),
        yes_side_share=('decisive_side', lambda s: (s == 'yes').mean()),
    )
    .reset_index()
    .sort_values(['research_category', 'confidence_slice', 'label_name'], kind='stable')
)
display(decisive_by_slice.head(20))


## Step 5: Score A Simple Baseline

Predictions for this release are keyed by `(market_id, cutoff_timestamp_utc)`.


In [ ]:
majority_label = int(train_target["label"].mode().iloc[0])
majority_label_name = train_target.loc[train_target["label"] == majority_label, "label_name"].mode().iloc[0]
median_hours = float(train_target["hours_to_decisive"].median())

test_predictions = test_target.loc[:, ["market_id", "cutoff_timestamp_utc"]].copy()
test_predictions["pred_label"] = majority_label
test_predictions["pred_hours_to_decisive"] = median_hours

evaluation = decisiveness_benchmark.evaluate(test_predictions, split="test")
print(f"Baseline ordinal prediction: {majority_label_name} ({majority_label})")
print(f"Baseline continuous prediction: {median_hours:.2f} hours")

display(evaluation["overall"])
display(evaluation["by_decisive_side"])
display(evaluation["continuous_overall"])


## Step 6: Sanity-Check The Split

The main lightweight checks are split sizes, ordinal label balance, and the amount of underlying market history behind each frozen prefix.


In [ ]:
split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "examples": len(train_df),
            "targets": len(train_target),
            "markets": train_df["market_id"].nunique(),
            "mean_hours_to_decisive": train_target["hours_to_decisive"].mean(),
            "median_prefix_rows": train_df["prefix_rows"].median(),
        },
        {
            "split": "test",
            "examples": len(test_df),
            "targets": len(test_target),
            "markets": test_df["market_id"].nunique(),
            "mean_hours_to_decisive": test_target["hours_to_decisive"].mean(),
            "median_prefix_rows": test_df["prefix_rows"].median(),
        },
    ]
)
display(split_summary)

label_balance = (
    decisiveness_benchmark.examples.groupby(["split", "label_name"], dropna=False)["market_id"]
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["split", "label_name"], kind="stable")
)
display(label_balance)

prefix_summary = (
    decisiveness_benchmark.market_timeseries.groupby("market_id", dropna=False)
    .size()
    .rename("history_rows")
    .reset_index()
    .merge(
        decisiveness_benchmark.examples
        .groupby(["market_id", "split", "decisive_side"], as_index=False)
        .agg(
            examples=("market_id", "size"),
            mean_hours_to_decisive=("hours_to_decisive", "mean"),
            mean_prefix_rows=("prefix_rows", "mean"),
        ),
        on="market_id",
        how="inner",
    )
)
display(prefix_summary.head(20))


## Step 7: Summarize The Release

These summaries are derived directly from the frozen release: one row per sampled prefix plus the normalized market histories behind those prefixes.


In [ ]:
release_audit = (
    decisiveness_benchmark.examples.groupby(["split", "decisive_side", "label_name"], dropna=False)
    .agg(
        rows=("market_id", "size"),
        markets=("market_id", "nunique"),
        mean_hours_to_decisive=("hours_to_decisive", "mean"),
        median_cutoff_age_hours=("cutoff_age_hours", "median"),
        mean_prefix_rows=("prefix_rows", "mean"),
        mean_current_yes_probability=("current_yes_probability", "mean"),
    )
    .reset_index()
    .sort_values(["split", "decisive_side", "label_name"], kind="stable")
)
display(release_audit)


## Step 8: Plot A Small Audit View

These plots use only the release artifact plus the explicit reference view you already built above.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

(
    decisiveness_benchmark.examples.groupby("split")["market_id"]
    .size()
    .plot(kind="bar", ax=axes[0], color=["#495057", "#C92A2A"])
)
axes[0].set_title("Decisive-belief examples by split")
axes[0].set_xlabel("split")
axes[0].set_ylabel("rows")

(
    decisiveness_view.groupby("confidence_slice")["hours_to_decisive"]
    .mean()
    .reindex(["50-60", "60-75", "75-90", "90-100"])
    .plot(kind="bar", ax=axes[1], color="#F08C00")
)
axes[1].set_title("Mean hours to decisiveness by confidence slice")
axes[1].set_xlabel("confidence slice")
axes[1].set_ylabel("hours")

plt.tight_layout()
plt.show()


## Step 9: Assess Sampling Cadence

The canonical history is already stored on an approximately 5-minute grid. The practical question here is whether the benchmark should also emit a new supervised example every 5 minutes, or whether a coarser cutoff cadence gives a cleaner benchmark.

The cells below estimate how many benchmark rows each cadence would create using the observed decisive windows in this frozen release, then measure how often a finer cadence could even change the ordinal target around the `24h` and `72h` label boundaries.


In [ ]:
cadence_by_market = (
    decisiveness_benchmark.examples.assign(
        decisive_age_hours=lambda df: df["cutoff_age_hours"] + df["hours_to_decisive"]
    )
    .groupby("market_id", as_index=False)
    .agg(
        decisive_age_hours=("decisive_age_hours", "max"),
        current_examples=("market_id", "size"),
    )
)

# Eligibility starts once the prefix is at least 6h old and still at least 1h away
# from the durable decisive event. The 24-point minimum history is only 2h on the
# 5-minute grid, so it is not the binding constraint here.
eligible_hours = (cadence_by_market["decisive_age_hours"] - 7.0).clip(lower=0.0)

cadence_steps_hours = {
    "5m": 1 / 12,
    "15m": 0.25,
    "30m": 0.5,
    "1h": 1.0,
    "3h": 3.0,
    "6h": 6.0,
    "12h": 12.0,
    "24h": 24.0,
}

cadence_audit = pd.DataFrame(
    [
        {
            "cadence": cadence,
            "estimated_rows": int((((eligible_hours / step_hours).astype(int)) + (eligible_hours > 0).astype(int)).sum()),
            "mean_examples_per_market": float((((eligible_hours / step_hours).astype(int)) + (eligible_hours > 0).astype(int)).mean()),
            "median_examples_per_market": float((((eligible_hours / step_hours).astype(int)) + (eligible_hours > 0).astype(int)).median()),
        }
        for cadence, step_hours in cadence_steps_hours.items()
    ]
)
cadence_audit["vs_current_release"] = cadence_audit["estimated_rows"] / len(decisiveness_benchmark.examples)
display(cadence_audit)

label_boundaries = BENCHMARK_CONFIG.ordinal_bin_edges_hours
boundary_sensitivity = pd.DataFrame(
    [
        {
            "cadence": cadence,
            "boundary_window_hours": step_hours,
            "rows_near_label_boundary": int(
                decisiveness_benchmark.examples["hours_to_decisive"].apply(
                    lambda hours: any(abs(float(hours) - float(boundary)) <= step_hours for boundary in label_boundaries)
                ).sum()
            ),
        }
        for cadence, step_hours in cadence_steps_hours.items()
    ]
)
boundary_sensitivity["share_near_label_boundary"] = (
    boundary_sensitivity["rows_near_label_boundary"] / len(decisiveness_benchmark.examples)
)
display(boundary_sensitivity)


## Step 10: Explicit Question And Answer

**Question.** Does it make sense to have probabilities every 5 minutes, or is it better to use a less frequent split?

**Answer.** The raw probability history can stay at 5-minute resolution, but the benchmark should not create a new supervised cutoff every 5 minutes. On this frozen release, moving from the current 12-hour cadence to a 5-minute cadence would expand the benchmark from `924,923` rows to about `137.8M` rows, roughly `149x` larger, and it would mostly add near-duplicate prefixes from the same markets. At the same time, only about `0.05%` of current examples sit within 5 minutes of the `24h` or `72h` label boundaries where the ordinal target could change, so the extra density would rarely change the task itself.

A less frequent split is better for this benchmark. If more temporal density is useful, `6h` is a reasonable next step because it would still be manageable at about `1.9M` rows, while `5m` is too fine for this target definition and would overweight long-lived markets without adding much new signal.
